In [1]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import pandas as pd

columns = ["Rank", "Team Name", "Total Earnings", "Total Tournaments Played"]

def get_team_details(row):
    cells = row.find_elements(By.TAG_NAME, "td")
    texts = [cell.text.strip() for cell in cells]
    
    # Remove texts after numbers in columns
    def clean_stat(text):
        return (
            text.replace("Tournaments", "")
                .replace("\n", " ")
                .strip()
        )


    return {
        "Rank": texts[0], 
        "Team Name": texts[1], 
        "Total Earnings": texts[2],              
        "Total Tournaments Played": clean_stat(texts[3])      
        
    }

def main():
    team_data = []

    driver = webdriver.Chrome()

    try:
        for page_id in range(1, 5):
            url = f"https://www.esportsearnings.com/teams?page={page_id}"
            driver.get(url)

            WebDriverWait(driver, 5).until(
                EC.presence_of_all_elements_located((By.CSS_SELECTOR, "table tbody tr"))
            )

            rows = driver.find_elements(By.CSS_SELECTOR, "table tbody tr")

            for row in rows:
                team = get_team_details(row)
                if team:
                    team_data.append(team)

                if len(team_data) >= 100:
                    break

            if len(team_data) >= 100:
                break

            for row in rows[:10]:
                cells = row.find_elements(By.TAG_NAME, "td")
                print(len(cells), [c.text for c in cells])


    finally:
        driver.quit()

    df = pd.DataFrame(team_data[:100], columns=columns)
    df.to_csv("top_teams_in_the_world.csv", index=False)

    print(df.head())
    print(f"Saved {len(df)} teams.")

if __name__ == "__main__":
    main()

  Rank      Team Name  Total Earnings Total Tournaments Played
0   1.    Team Liquid  $56,790,405.44                     2987
1   2.             OG  $38,773,813.62                      208
2   3.    Team Spirit  $35,719,420.44                      311
3   4.  Evil Geniuses  $28,567,423.99                     1019
4   5.  Natus Vincere  $24,522,553.68                      874
Saved 100 teams.
